# Semantic Kernel Client Summarization (Chunked Notes)

This notebook ports `notebooks/summarization.ipynb` to Semantic Kernel and summarizes large notes in chunks to reduce context length.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase
try:
    from semantic_kernel.contents.chat_history import ChatHistory
except ImportError:
    from semantic_kernel.contents import ChatHistory

In [2]:
load_dotenv('/home/azureuser/cloudfiles/code/email-generation/.env.local', override=True)

AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_LANGUAGE_ENDPOINT')
AZURE_API_VERSION = os.getenv('AZURE_API_VERSION')
AZURE_OPENAI_DEPLOYMENT_NAME = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')

kernel = Kernel()
service = AzureChatCompletion(
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_API_VERSION,
)
kernel.add_service(service)

chat_service = kernel.get_service(type=ChatCompletionClientBase)

In [3]:
PREFIX = '/home/azureuser/cloudfiles/code/production-data/12-dec-25/tenant1-12-20-25'

# invoices.csv: 100 samples
# ['client_id', 'matter_id', 'invoice_number', 'invoice_amount', 'ar_amount', 'submitted_on', 'due_on', 'days_overdue', 'status']
df_invoices = pd.read_csv(f'{PREFIX}-invoices.csv')

# activities.csv
# ['client_id', 'matter_id', 'invoice_number', 'created_on', 'event_type', 'name', 'description']
df_activities = pd.read_csv(
    f'{PREFIX}-activities-REDACTED.csv',
    parse_dates=['created_on'],
    date_format='%Y-%m-%d %H:%M:%S.%f%z'
)

# notes.csv
# ['client_id', 'matter_id', 'invoice_number', 'created_on', 'content']
df_notes = pd.read_csv(f'{PREFIX}-notes-REDACTED.csv', parse_dates=['created_on'])
df_notes['created_on'] = pd.to_datetime(df_notes['created_on'], format='mixed')
df_notes = df_notes.dropna()
df_notes['invoice_number'] = df_notes['invoice_number'].apply(lambda x: int(x))

In [4]:
def data_formatter_csv_list(df_invoices, df_notes, df_activities, invoice_numbers):
    if not isinstance(invoice_numbers, list):
        invoice_numbers = [invoice_numbers]

    invoice_data = (
        df_invoices.query('invoice_number in @invoice_numbers')
        .drop(columns=['client_id', 'matter_id'])
        .to_csv(index=False)
    )

    activities = (
        df_activities.query('invoice_number in @invoice_numbers')
        .dropna()
        .drop(columns=['client_id', 'matter_id'])
        .to_csv(index=False)
    )

    invoice_numbers_int = [int(str(inv_num).split('-')[0]) for inv_num in invoice_numbers]
    notes = (
        df_notes.query('invoice_number in @invoice_numbers_int')
        [['invoice_number', 'created_on', 'content']]
        .to_csv(index=False)
    )

    formatted_text = f'''
### INVOICE DATA:
{invoice_data}

### NOTES:
{notes}

### ACTIVITIES:
{activities}
'''
    return formatted_text, notes, invoice_data, activities


def build_invoice_groups(df_invoices):
    invoice_numbers = df_invoices.groupby(by='matter_id').agg(list).invoice_number.tolist()
    return [lst for lst in invoice_numbers if len(lst) >= 2]

In [5]:
SYSTEM_PROMPT = '''
You are Oddr Summarizer, an AI assistant that helps law firms manage their Accounts Receivable. Your job is to generate clear, concise, and helpful summaries for attorneys using only the facts provided.

You may infer logical next steps or patterns (e.g., payment gaps, lack of follow-up), but do not fabricate or invent any client interactions or internal updates that are not explicitly present in the input.

If no notes are available, avoid referring to interactions. Focus only on AR status and suggest possible outreach or monitoring based on the financial data.

Your tone should be professional, factual, and efficient, like a trusted human AR professional writing a briefing. Favor direct, punchy sentences over formal or padded language.

Given a client with the following financial data and notes across the client, its matters, and invoices, please summarize the client in a way that will make it digestible and meaningful to the time-constrained Attorney responsible for the client so they can understand what is going on with their AR and who is responsible for any next steps.

Frame this as a timeline, starting with an overview of the client's current AR status, then highlighting the most recent interaction and any key themes from earlier interactions, but only if such notes are present. Do not fabricate or infer specific updates. However, it is acceptable to suggest likely follow-up steps or highlight the absence of recent activity if that pattern is evident.

Reduce repetitive details by grouping similar updates, eliminate specific payment amounts where appropriate, and keep the summary focused and efficient. Mention past events only if they are essential to understanding the current delay or next step. Avoid repeating older context that does not affect present action.

Format the summary as a single paragraph without any headings or titles, and use smart **bolding** to highlight key points.
Limit the summary to **4 complete, efficient sentences**. Favor punchy, high-signal writing over formal explanations.
**Do not use headings, bullet points, or labels.** **Mention date always like Jan 01,2025**

Structure the summary as a single flowing paragraph that follows this order:
[AR overview]
[Highlights from recent interactions - only if present]
[Relevant context from earlier interactions - only if helpful]
[Next steps or follow-up actions]
'''.strip()

CHUNK_SUMMARY_PROMPT = '''
You are summarizing a chunk of client notes to reduce context length before final summarization.
- Keep dates, invoice numbers, amounts, and action/status updates.
- Preserve the most recent developments.
- Group repetitive updates.
- Do not invent events or add commentary.
Return 3-4 short sentences. No headings or bullet points.
'''.strip()

MERGE_CHUNKS_PROMPT = '''
Combine the chunk summaries into a single concise timeline for an attorney.
- Keep dates, invoice numbers, and action/status updates.
- Preserve the most recent developments.
- Remove repetition.
Return 6-8 short sentences. No headings or bullet points.
'''.strip()

In [6]:
def split_notes_csv(notes_csv, max_lines=40):
    lines = [line.strip() for line in notes_csv.splitlines() if line.strip()]
    if lines and 'invoice_number' in lines[0].lower():
        lines = lines[1:]
    if not lines:
        return []
    return ['\n'.join(lines[i:i+max_lines]) for i in range(0, len(lines), max_lines)]

In [7]:
async def chat_complete(system_prompt, user_prompt, max_tokens=400, temperature=0.2):
    chat_history = ChatHistory()
    chat_history.add_system_message(system_prompt)
    chat_history.add_user_message(user_prompt)

    settings_cls = chat_service.get_prompt_execution_settings_class()
    settings = settings_cls(max_tokens=max_tokens, temperature=temperature)

    result = await chat_service.get_chat_message_contents(
        chat_history=chat_history,
        settings=settings,
        kernel=kernel,
    )
    if not result:
        return ''
    return result[0].content or ''


async def summarize_notes_in_chunks(notes_csv, max_lines=40):
    chunks = split_notes_csv(notes_csv, max_lines=max_lines)
    if not chunks:
        return ''

    chunk_summaries = []
    for idx, chunk in enumerate(chunks, start=1):
        user_prompt = f'Chunk {idx}/{len(chunks)}:\n{chunk}'
        summary = await chat_complete(
            CHUNK_SUMMARY_PROMPT,
            user_prompt,
            max_tokens=300,
            temperature=0.2,
        )
        chunk_summaries.append(summary.strip())

    if len(chunk_summaries) == 1:
        return chunk_summaries[0]

    merge_prompt = '\n\n'.join(chunk_summaries)
    merged = await chat_complete(
        MERGE_CHUNKS_PROMPT,
        merge_prompt,
        max_tokens=400,
        temperature=0.2,
    )
    return merged.strip()

In [8]:
async def generate_client_summary(financial_data, notes_summary):
    user_message = (
        'Financial data:\n'
        f'{financial_data}\n\n'
        'Notes (summarized for context):\n'
        f'{notes_summary}'
    )

    return await chat_complete(
        SYSTEM_PROMPT,
        user_message,
        max_tokens=600,
        temperature=0.4,
    )

In [9]:
invoice_groups = build_invoice_groups(df_invoices)

group_index = 32
invoice_numbers = invoice_groups[group_index]

_, notes_csv, invoice_data, _ = data_formatter_csv_list(
    df_invoices,
    df_notes,
    df_activities,
    invoice_numbers,
)

notes_summary = await summarize_notes_in_chunks(notes_csv, max_lines=40)
summary = await generate_client_summary(invoice_data, notes_summary)

notes_summary, summary

('',
 "The client's current Accounts Receivable (AR) stands at **$34,240.50**, with one invoice overdue by **36 days** and another due in **7 days**. The most recent invoice, submitted on **Dec 10, 2025**, is currently **delivered** and due on **Jan 09, 2026**. There have been no recent follow-ups noted regarding the overdue invoice, which may require immediate attention to avoid further delays. It is advisable to reach out to the client regarding the overdue amount and confirm receipt of the latest invoice to ensure timely payment.")